# AGM-031 — Official CV evaluation (Google Colab)

This notebook orchestrates the repository-owned two-phase scientific pipeline. It targets a Tesla T4, uses Python 3.11 and the pinned AGM-030 dependencies, and never duplicates training, threshold, metric, or TEST logic.

**Safety:** normal `Run all` execution stops at the explicit TEST gate. TEST has never been opened.

In [ ]:
# A. Runtime and GPU diagnostics — fail closed before setup.
import os, platform, subprocess

subprocess.run(["nvidia-smi"], check=True)
gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip().splitlines()[0]
assert gpu_name == "Tesla T4", f"Official AGM-031 hardware must be Tesla T4, got {gpu_name!r}"
print({"colab_system_python": platform.python_version(), "gpu": gpu_name})

## Repository authentication

The repository is public by default. If it becomes private, create a Colab secret named `GITHUB_TOKEN`, enable notebook access to it, and set `PRIVATE_REPOSITORY = True` below. The token is used through a temporary `GIT_ASKPASS` helper, is never printed, and is deleted immediately after cloning.

In [ ]:
# B–D. Clone and verify the exact scientific revision.
from pathlib import Path
import stat, tempfile

REPOSITORY_URL = "https://github.com/waelzouari/AgriMind.git"
SCIENTIFIC_REVISION = "22bc30ef3b350e4bffe0386eb11dcb7aa79345b7"
REPOSITORY = Path("/content/AgriMind")
PRIVATE_REPOSITORY = False
assert not REPOSITORY.exists(), "/content/AgriMind already exists; use a fresh Colab runtime"

environment = os.environ.copy()
askpass = None
try:
    if PRIVATE_REPOSITORY:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        assert token, "GITHUB_TOKEN is unavailable in Colab Secrets"
        handle = tempfile.NamedTemporaryFile(mode="w", delete=False, prefix="agm031-askpass-")
        handle.write("#!/bin/sh\nprintf '%s\n' \"$GITHUB_TOKEN\"\n")
        handle.close()
        askpass = Path(handle.name)
        askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
        environment.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "GITHUB_TOKEN": token})
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY)], check=True, env=environment)
finally:
    environment.pop("GITHUB_TOKEN", None)
    if askpass is not None:
        askpass.unlink(missing_ok=True)

subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach", SCIENTIFIC_REVISION], check=True)
observed = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True).strip()
assert observed == SCIENTIFIC_REVISION
assert not subprocess.check_output(["git", "-C", str(REPOSITORY), "status", "--porcelain"], text=True).strip()
os.chdir(REPOSITORY)
print("Scientific revision verified:", observed)

In [ ]:
# B–C. Create an isolated Python 3.11 environment and install exact scientific dependencies.
MICROMAMBA = Path("/content/bin/micromamba")
ENV_PREFIX = Path("/content/agm031-python311")
if not MICROMAMBA.exists():
    subprocess.run("mkdir -p /content/bin && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/bin --strip-components=1 bin/micromamba", shell=True, check=True)
subprocess.run([str(MICROMAMBA), "create", "-y", "-p", str(ENV_PREFIX), "python=3.11", "pip"], check=True)
PYTHON = str(ENV_PREFIX / "bin/python")
subprocess.run([PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "--index-url", "https://download.pytorch.org/whl/cu124", "torch==2.5.1", "torchvision==0.20.1"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "-c", "ai/computer_vision/training-constraints.txt", "-e", "./ai/computer_vision[train]"], check=True)

In [ ]:
# E–F. Acquire only the approved immutable source and verify its archive SHA-256.
ARCHIVE_SHA256 = "ae430d573bda45084545c84d5ca54a830d932962e80f385396956d799d1f254e"
ARCHIVE = REPOSITORY / "ai/computer_vision/data/raw/PlantVillage-Dataset-825c387b026b01570caa85ab7fdba5cb594adab0.zip"
DATASET_ROOT = REPOSITORY / "ai/computer_vision/data/raw/extracted/PlantVillage-Dataset-825c387b026b01570caa85ab7fdba5cb594adab0"  # pragma: allowlist secret
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["curl", "--fail", "--location", "--retry", "5", "--retry-delay", "5", "-o", str(ARCHIVE), "https://github.com/gabrieldgf4/PlantVillage-Dataset/archive/825c387b026b01570caa85ab7fdba5cb594adab0.zip"], check=True)
import hashlib
digest = hashlib.sha256()
with ARCHIVE.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == ARCHIVE_SHA256, "Approved archive SHA-256 mismatch"
DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["unzip", "-q", str(ARCHIVE), "-d", str(DATASET_ROOT.parent)], check=True)
assert DATASET_ROOT.is_dir()
print("Approved archive verified:", ARCHIVE_SHA256)

In [ ]:
# Environment versions and CUDA strategy — no CPU fallback.
diagnostics = '''
import json, platform, numpy, PIL, torch, torchvision
assert platform.python_version().startswith("3.11.")
assert numpy.__version__ == "2.1.3"
assert PIL.__version__ == "11.0.0"
assert torch.__version__.split('+')[0] == "2.5.1"
assert torchvision.__version__.split('+')[0] == "0.20.1"
assert torch.cuda.is_available()
assert torch.cuda.get_device_name(0) == "Tesla T4"
print(json.dumps({"python": platform.python_version(), "numpy": numpy.__version__, "pillow": PIL.__version__, "torch": torch.__version__, "torchvision": torchvision.__version__, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(), "gpu": torch.cuda.get_device_name(0), "gpu_vram_bytes": torch.cuda.get_device_properties(0).total_memory}, indent=2))
'''
subprocess.run([PYTHON, "-c", diagnostics], check=True)

In [ ]:
# G–Q. Audit, fingerprint gates, split/leakage gates, CUDA TRAIN, VALIDATION threshold, and freeze. TEST is not read.
COMMON = [
    "--dataset-config", "ai/computer_vision/configs/datasets/plantvillage-notebook-mirror-v1.json",
    "--training-config", "ai/computer_vision/configs/training/mobilenet-v2-v1.json",
    "--official-config", "ai/computer_vision/configs/evaluation/mobilenet-v2-v1.json",
    "--dataset-root", str(DATASET_ROOT.relative_to(REPOSITORY)),
    "--source-archive", str(ARCHIVE.relative_to(REPOSITORY)),
    "--audit-report", "ai/computer_vision/data/interim/agm-031-audit.json",
    "--related-manifest", "ai/computer_vision/data/interim/agm-031-related.jsonl",
    "--split-manifest", "ai/computer_vision/data/interim/agm-031-split.jsonl",
    "--model", "ai/computer_vision/artifacts/agrimind-cv-mobilenet-v2-v1.pt",
    "--pretest-evidence", "ai/computer_vision/artifacts/agm-031-pretest.json",
    "--runtime-manifest", "ai/computer_vision/evaluation/mobilenet-v2-v1.runtime.json",
    "--evaluation-report", "ai/computer_vision/evaluation/mobilenet-v2-v1.json",
    "--test-opening-marker", "ai/computer_vision/artifacts/agm-031-test-opened.txt",
]
subprocess.run([PYTHON, "-u", "-m", "agrimind_cv.cli", "official-prepare", *COMMON], check=True)

In [ ]:
# R. HUMAN TEST-OPENING CHECKPOINT. Normal Run all MUST stop here.
import json
pretest = json.loads(Path("ai/computer_vision/artifacts/agm-031-pretest.json").read_text())
manifest = json.loads(Path("ai/computer_vision/artifacts/agrimind-cv-mobilenet-v2-v1.runtime.json").read_text())
summary = {
    "dataset_fingerprint": pretest["dataset"]["fingerprint"],
    "related_manifest_fingerprint": pretest["dataset"]["related_manifest_fingerprint"],
    "split_fingerprint": pretest["split"]["fingerprint"],
    "validation_threshold": pretest["validation"]["threshold"],
    "artifact_sha256": pretest["artifact"]["sha256"],
    **pretest["freeze"],
}
print(json.dumps(summary, indent=2))
assert pretest["freeze"] == {"model_frozen": True, "preprocessing_frozen": True, "threshold_frozen": True, "test_opened": False}
assert not Path("ai/computer_vision/artifacts/agm-031-test-opened.txt").exists()
OPEN_OFFICIAL_TEST = False  # Manually change to True only after reviewing the summary above.
assert OPEN_OFFICIAL_TEST is True, "TEST REMAINS SEALED. Review evidence, then explicitly set OPEN_OFFICIAL_TEST = True."

In [ ]:
# S–V. One-time held-out TEST evaluation. This cell independently rechecks the manual flag.
assert globals().get("OPEN_OFFICIAL_TEST") is True, "Explicit TEST authorization is absent"
assert not Path("ai/computer_vision/artifacts/agm-031-test-opened.txt").exists(), "TEST was already opened"
subprocess.run([PYTHON, "-u", "-m", "agrimind_cv.cli", "official-test", *COMMON], check=True)
assert Path("ai/computer_vision/artifacts/agm-031-test-opened.txt").is_file()
print("Official held-out TEST evaluation completed once.")

In [ ]:
# W–X. Create the minimal download bundle; no dataset, images, caches, credentials, or sample lists.
import shutil, zipfile
bundle_root = Path("/content/agm-031-official-output")
bundle_root.mkdir(exist_ok=False)
outputs = [
    Path("ai/computer_vision/artifacts/agrimind-cv-mobilenet-v2-v1.pt"),
    Path("ai/computer_vision/artifacts/agm-031-test-opened.txt"),
    Path("ai/computer_vision/evaluation/mobilenet-v2-v1.runtime.json"),
    Path("ai/computer_vision/evaluation/mobilenet-v2-v1.json"),
]
for source in outputs:
    assert source.is_file(), f"Missing official output: {source}"
    shutil.copy2(source, bundle_root / source.name)
zip_path = Path("/content/agm-031-official-output.zip")
with zipfile.ZipFile(zip_path, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for source in sorted(bundle_root.iterdir()):
        archive.write(source, arcname=source.name)
print("Download-ready bundle:", zip_path)

In [ ]:
# Download only after the official run has completed successfully.
from google.colab import files
files.download("/content/agm-031-official-output.zip")